# Plate Raster + Synchrony Viewer

Visualizes burst detection results as a 4×6 plate grid showing raster plots overlaid with synchrony signals (sharp + smooth) for each well.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from pipeline_manager import PipelineManager, WorkItem
from pipeline_manager.task_record import TaskStatus
from config_manager import ConfigManager
from pipeline_tasks import (
    PreprocessingTask,
    SortingTask,
    AutoMergeTask,
    AnalyzerTask,
    AutoCurationTask,
    BurstDetectionTask,
    PlateViewerTask,
)


In [ ]:
# Config setup — generates template if it doesn't exist
CONFIG_FILE = Path("../config/default_tasks_params_01_plateviewer.json")

UPSTREAM_TASK_CLASSES = [
    PreprocessingTask,
    SortingTask,
    AutoMergeTask,
    AnalyzerTask,
    AutoCurationTask,
    BurstDetectionTask,
]
ALL_TASK_CLASSES = UPSTREAM_TASK_CLASSES + [PlateViewerTask]

cm = ConfigManager()
cm.set_global("analysis_root", "/path/to/analysis")
for cls in ALL_TASK_CLASSES:
    cm.register_task(cls)

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Config template written to {CONFIG_FILE}.\n"
        "Edit it to set analysis_root, burst_detection_root, curation_output_root, figures_root, etc.\n"
        "Then re-run this cell."
    )

cm.load(CONFIG_FILE)
ANALYSIS_ROOT = Path(cm.get_global("analysis_root"))
print(f"Config loaded from {CONFIG_FILE}")
print(f"  analysis_root: {ANALYSIS_ROOT}")


In [ ]:
# Select recording to visualize
RECORDING_KEY = "CX138/260329/T003346/Network/000029"
print(f"Recording: {RECORDING_KEY}")

In [ ]:
# Register and run plate viewer task
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)
pipeline_mgr = PipelineManager(ANALYSIS_ROOT, config_provider=cm)

for cls in ALL_TASK_CLASSES:
    try:
        pipeline_mgr.register_task(cls)
        print(f"Registered task: {cls.task_name!r}")
    except ValueError as e:
        print(f"Task already registered ({e}) — continuing")

# Plate-level task: registered once with sentinel well_id.
pipeline_mgr.add_well(RECORDING_KEY, "__plate__")

# The per-well burst_detection tasks are the real upstream work. The sentinel
# entry only exists to schedule the plate-level viewer after those outputs exist.
for cls in UPSTREAM_TASK_CLASSES:
    dep_item = WorkItem(RECORDING_KEY, "__plate__", cls.task_name)
    pipeline_mgr.update_status(dep_item, TaskStatus.COMPLETE)

pending = [
    item
    for item in pipeline_mgr.get_next_task(n=50, recording_keys={RECORDING_KEY})
    if item.task_name == PlateViewerTask.task_name and item.well_id == "__plate__"
]

if not pending:
    print("No pending plate_viewer task.")
else:
    task = PlateViewerTask()
    for item in pending:
        print(f"Running {item.task_name} for {item.recording_key} ...")
        params = cm.get_config(item.task_name, item.recording_key, item.well_id)
        pipeline_mgr.update_status(item, TaskStatus.RUNNING)
        try:
            output = task.run(item.recording_key, item.well_id, data_path=Path("."), params=params)
            pipeline_mgr.update_status(item, TaskStatus.COMPLETE, output_path=output)
            print(f"✓ Saved: {output}")
        except Exception as exc:
            pipeline_mgr.update_status(item, TaskStatus.FAILED, error=str(exc))
            raise


In [ ]:
# Display the generated HTML
from IPython.display import IFrame

figures_root = Path(cm.get_task_params("plate_viewer").get("figures_root", "./figures"))
html_path = figures_root / RECORDING_KEY / "plate_viewer.html"

if html_path.exists():
    print(f"Displaying: {html_path}")
    IFrame(src=str(html_path), width="100%", height=800)
else:
    print(f"HTML not found: {html_path}")